In [6]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START,END
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import AnyMessage, AIMessage,BaseMessage


In [7]:
load_dotenv()

True

In [8]:
llm=ChatGroq(model='openai/gpt-oss-120b')

In [9]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [10]:
def chat_node(state: ChatState):

    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question.",
        "question": state["messages"][-1].content,
        "instruction": "Approve this question? yes/no"
    })
    
    if decision["approved"] == 'no':
        return {"messages": [AIMessage(content="Not approved.")]}

    else:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

In [11]:
# 3. Build the graph: START -> chat -> END
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Checkpointer is required for interrupts
checkpointer = MemorySaver()

# Compile the app
app = builder.compile(checkpointer=checkpointer)

In [12]:
# Create a new thread id for this conversation
config = {"configurable": {"thread_id": '1234'}}

# ---- STEP 1: user asks a question ----
initial_input = {
    "messages": [
        ("user", "Explain gradient descent in very simple terms.")
    ]
}

# Invoke the graph for the first time
result = app.invoke(initial_input, config=config)

In [13]:
result

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='3d9eac7b-ca89-44a8-88b6-f0302d7ce574')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer a user question.', 'question': 'Explain gradient descent in very simple terms.', 'instruction': 'Approve this question? yes/no'}, id='98a9f4fa3a0a4023f9feec8dc44b680d')]}

In [14]:
message = result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'Explain gradient descent in very simple terms.',
 'instruction': 'Approve this question? yes/no'}

In [15]:

user_input = input(f"\nBackend message - {message} \n Approve this question? (y/n): ")

In [16]:
# Resume the graph with the approval decision
final_result = app.invoke(
    Command(resume={"approved": user_input}),
    config=config,
)

In [17]:
print(final_result["messages"][-1].content)

**Gradient descent is a way for a computer (or a person) to find the lowest point of a bumpy surface, like a valley in a hilly landscape.**  

Imagine you’re standing on a foggy hill and you can’t see very far. All you can feel is the slope under your feet. If you want to get to the bottom of the valley, you would:

1. **Feel which way the ground slopes downhill.**  
   That direction is the *gradient* – a fancy word for “the slope” or “the steepest direction of change.”

2. **Take a small step in that downhill direction.**  
   The step size is called the *learning rate*. If you step too far you might overshoot the valley; if you step too tiny you’ll take forever.

3. **Repeat:**  
   After each step, feel the new slope, step downhill again, and keep going until the slope feels flat (i.e., you’re at or very near the bottom).

In mathematics we do the same thing, but instead of a physical hill we have a *cost* or *error* function that measures how wrong a model’s predictions are. The s